In [6]:
import pandas as pd
train_df=pd.read_csv('../data/train_df.tsv', sep='\t', header=None, names=['impression_id',	'user_id',	'time',	'history',	'impressions',	'datetime'])
test_df=pd.read_csv('../data/test_df.tsv', sep='\t', header=None, names=['impression_id',	'user_id',	'time',	'history',	'impressions',	'datetime'])

In [7]:
news_df=pd.read_csv('../data/news.tsv', sep='\t', header=None, names=['news_id', 'category', 'subcategory', 'title', 'abstract', 'url', 'title_entities', 'abstract_entities'])
news_df

,news_id,category,subcategory,title,abstract,url,title_entities,abstract_entities
0,N55528,lifestyle,lifestyleroyals,"The Brands Queen Elizabeth, Prince Charles, an...","Shop the notebooks, jackets, and more that the...",https://assets.msn.com/labs/mind/AAGH0ET.html,"[{""Label"": ""Prince Philip, Duke of Edinburgh"",...",[]
1,N19639,health,weightloss,50 Worst Habits For Belly Fat,These seemingly harmless habits are holding yo...,https://assets.msn.com/labs/mind/AAB19MK.html,"[{""Label"": ""Adipose tissue"", ""Type"": ""C"", ""Wik...","[{""Label"": ""Adipose tissue"", ""Type"": ""C"", ""Wik..."
2,N61837,news,newsworld,The Cost of Trump's Aid Freeze in the Trenches...,Lt. Ivan Molchanets peeked over a parapet of s...,https://assets.msn.com/labs/mind/AAJgNsz.html,[],"[{""Label"": ""Ukraine"", ""Type"": ""G"", ""WikidataId..."
3,N53526,health,voices,I Was An NBA Wife. Here's How It Affected My M...,"I felt like I was a fraud, and being an NBA wi...",https://assets.msn.com/labs/mind/AACk2N6.html,[],"[{""Label"": ""National Basketball Association"", ..."
4,N38324,health,medical,"How to Get Rid of Skin Tags, According to a De...","They seem harmless, but there's a very good re...",https://assets.msn.com/labs/mind/AAAKEkt.html,"[{""Label"": ""Skin tag"", ""Type"": ""C"", ""WikidataI...","[{""Label"": ""Skin tag"", ""Type"": ""C"", ""WikidataI..."
...,...,...,...,...,...,...,...,...
51277,N16909,weather,weathertopstories,"Adapting, Learning And Soul Searching: Reflect...",Woolsey Fire Anniversary: A community is forev...,https://assets.msn.com/labs/mind/BBWzQJK.html,"[{""Label"": ""Woolsey Fire"", ""Type"": ""N"", ""Wikid...","[{""Label"": ""Woolsey Fire"", ""Type"": ""N"", ""Wikid..."
51278,N47585,lifestyle,lifestylefamily,Family says 13-year-old Broadway star died fro...,NaN,https://assets.msn.com/labs/mind/BBWzQYV.html,"[{""Label"": ""Broadway theatre"", ""Type"": ""F"", ""W...",[]
51279,N7482,sports,more_sports,St. Dominic soccer player tries to kick cancer...,"Sometimes, what happens on the sidelines can b...",https://assets.msn.com/labs/mind/BBWzQnK.html,[],[]
51280,N34418,sports,soccer_epl,How the Sounders won MLS Cup,"Mark, Jeremiah and Casey were so excited they ...",https://assets.msn.com/labs/mind/BBWzQuK.html,"[{""Label"": ""MLS Cup"", ""Type"": ""U"", ""WikidataId...",[]


In [8]:
import numpy as np
from collections import Counter

# 1. Считаем глобальную популярность и свежесть
def extract_item_stats(df):
    item_clicks = Counter()
    item_impressions = Counter()

    for imp in df['impressions'].dropna():
        items = str(imp).split()
        for item in items:
            parts = item.rsplit('-', 1)  # rsplit с конца на 2 части на случай, если в news_id есть '-'
            if len(parts) == 2:
                news_id, label_str = parts[0], parts[1]
                try:
                    label = int(label_str)
                    item_impressions[news_id] += 1
                    if label == 1:
                        item_clicks[news_id] += 1
                except ValueError:
                    # Если за дефисом шло не число
                    continue
            else:
                # В строке нет дефиса с меткой
                continue

    return item_clicks, item_impressions

train_clicks, train_impressions = extract_item_stats(train_df)

# 2. Функция сборки табличного датасета (Unrolling impressions)
def prepare_ranking_dataset(df, is_train=True, max_samples=None):
    rows = []

    if max_samples:
        df = df.head(max_samples)

    for idx, row in df.iterrows():
        imp_str = row['impressions']
        if pd.isna(imp_str):
            continue

        history = str(row['history']).split() if pd.notna(row['history']) else []
        history_len = len(history)

        items = str(imp_str).split()
        for pos, item in enumerate(items):
            parts = item.rsplit('-', 1)

            # Извлекаем news_id и метку target
            if len(parts) == 2:
                news_id, label_str = parts[0], parts[1]
                try:
                    label = int(label_str)
                except ValueError:
                    label = np.nan
            else:
                news_id = parts[0]
                label = np.nan  # Если меток нет (для чисто тестовых данных без граунд трут)

            # Генерация CTR и показателей популярности из Train
            clicks = train_clicks.get(news_id, 0)
            imps = train_impressions.get(news_id, 0)
            ctr = clicks / imps if imps > 0 else 0.0

            rows.append({
                'session_id': idx,
                'news_id': news_id,
                'position_in_impression': pos,
                'history_len': history_len,
                'train_clicks': clicks,
                'train_impressions': imps,
                'train_ctr': ctr,
                'target': label
            })

    return pd.DataFrame(rows)

print("Сборка обучающего датасета...")
X_train_df = prepare_ranking_dataset(train_df, is_train=True)
print(f"Обучающих строк (пара сессия-новость): {len(X_train_df)}")

print("Сборка тестового датасета...")
X_test_df = prepare_ranking_dataset(test_df, is_train=False)
print(f"Тестовых строк: {len(X_test_df)}")

Сборка обучающего датасета...
Обучающих строк (пара сессия-новость): 4621016
Сборка тестового датасета...
Тестовых строк: 1222430


In [9]:
from catboost import CatBoostClassifier
from collections import Counter
import tqdm


print(f"Строк в X_train_df ДО очистки: {len(X_train_df)}")

# 1. Проверяем наличие NaN в таргете
nans_in_target = X_train_df['target'].isnull().sum()
print(f"Найдено NaN значений в колонке 'target': {nans_in_target}")

# 2. Если есть пропуски, удаляем эти строки
if nans_in_target > 0:
    print("Удаляем строки с NaN в таргете...")
    # inplace=True заменяет датафрейм, subset=['target'] указывает колонку
    X_train_df.dropna(subset=['target'], inplace=True)
    print(f"Строк в X_train_df ПОСЛЕ очистки: {len(X_train_df)}")
else:
    print("Пропусков в таргете не найдено.")

# Дополнительная проверка на всякий случай (иногда таргет может быть не int)
X_train_df['target'] = X_train_df['target'].astype(int)


# 1. Подготовка данных для CatBoost
# Фичи, которые мы будем использовать (пока только числовые)
feature_cols = ['position_in_impression', 'history_len', 'train_clicks', 'train_impressions', 'train_ctr']

X_train = X_train_df[feature_cols]
y_train = X_train_df['target']

X_test = X_test_df[feature_cols]
y_test = X_test_df['target']

# Рассчитываем веса для балансировки классов (т.к. CTR ~4%)
# Расчет: кол-во_отрицательных / кол-во_положительных
pos_count = y_train.sum()
neg_count = len(y_train) - pos_count
scale_weight = neg_count / pos_count
print(f"Классы: {neg_count} (0) / {pos_count} (1). scale_pos_weight = {scale_weight:.2f}")

# 2. Обучение модели
print("Начинаем обучение CatBoostClassifier...")
# Ограничим параметры для скорости пет-проекта, в реальном проекте ставим больше
model = CatBoostClassifier(
    iterations=500,        # кол-во деревьев
    depth=6,               # глубина деревьев
    learning_rate=0.1,     # скорость обучения
    scale_pos_weight=scale_weight, # балансировка
    loss_function='Logloss',
    verbose=100,           # вывод логов каждые 100 итераций
    random_seed=42
)

model.fit(X_train, y_train)
print("Модель обучена!")

# 3. Инференс на тесте (получаем вероятности клика)
print("Прогнозирование на тестовых данных...")
preds_proba = model.predict_proba(X_test)[:, 1] # Вероятность класса 1
X_test_df['score'] = preds_proba

# 4. Расчет метрик ранжирования (по сессиям)
def recall_at_k(actual, predicted, k=5):
    predicted_at_k = predicted[:k]
    hits = len(set(actual) & set(predicted_at_k))
    return hits / len(actual) if len(actual) > 0 else 0.0

def ndcg_at_k(actual, predicted, k=5):
    predicted_at_k = predicted[:k]
    dcg = 0.0
    for i, p in enumerate(predicted_at_k):
        if p in actual:
            dcg += 1.0 / np.log2(i + 2)
    idcg = sum([1.0 / np.log2(i + 2) for i in range(min(len(actual), k))])
    return dcg / idcg if idcg > 0 else 0.0

model_ndcg, model_recall = [], []

# Группируем тестовый датасет по сессиям, чтобы отранжировать кандидатов
test_sessions = X_test_df.groupby('session_id')

print("Расчет метрик ранжирования для модели...")
for _, session_data in tqdm.tqdm(test_sessions):
    actual_clicks = session_data[session_data['target'] == 1]['news_id'].tolist()

    if not actual_clicks:
        continue # Пропускаем сессии без кликов (хотя в MIND их почти нет в impressions)

    # Сортируем кандидатов по скору модели (от большего к меньшему)
    ranked_recommendations = session_data.sort_values(by='score', ascending=False)['news_id'].tolist()

    model_ndcg.append(ndcg_at_k(actual_clicks, ranked_recommendations, k=5))
    model_recall.append(recall_at_k(actual_clicks, ranked_recommendations, k=5))

print("\n=== СРАВНЕНИЕ РЕЗУЛЬТАТОВ (на Test set) ===")
print(f"1. Random Baseline:    NDCG@5 = 0.2028 | Recall@5 = 0.3078")
print(f"2. Popularity Baseline: NDCG@5 = 0.2168 | Recall@5 = 0.3253")
print(f"3. CatBoost (baseline features): NDCG@5 = {np.mean(model_ndcg):.4f} | Recall@5 = {np.mean(model_recall):.4f}")

Строк в X_train_df ДО очистки: 4621016
Найдено NaN значений в колонке 'target': 1
Удаляем строки с NaN в таргете...
Строк в X_train_df ПОСЛЕ очистки: 4621015
Классы: 4431496 (0) / 189519 (1). scale_pos_weight = 23.38
Начинаем обучение CatBoostClassifier...
0:	learn: 0.6735817	total: 338ms	remaining: 2m 48s
100:	learn: 0.5801916	total: 38.8s	remaining: 2m 33s
200:	learn: 0.5789861	total: 1m 9s	remaining: 1m 42s
300:	learn: 0.5779013	total: 1m 40s	remaining: 1m 6s
400:	learn: 0.5769962	total: 2m 11s	remaining: 32.4s
499:	learn: 0.5762003	total: 2m 42s	remaining: 0us
Модель обучена!
Прогнозирование на тестовых данных...
Расчет метрик ранжирования для модели...


100%|██████████| 30270/30270 [00:14<00:00, 2142.19it/s]


=== СРАВНЕНИЕ РЕЗУЛЬТАТОВ (на Test set) ===
1. Random Baseline:    NDCG@5 = 0.2028 | Recall@5 = 0.3078
2. Popularity Baseline: NDCG@5 = 0.2168 | Recall@5 = 0.3253
3. CatBoost (baseline features): NDCG@5 = 0.2319 | Recall@5 = 0.3408


In [12]:
# --- ПОДГОТОВКА СЛОВАРЕЙ ДЛЯ БЫСТРОГО ПОИСКА ---
print("Подготовка вспомогательных словарей...")

# 1. Словарь категорий новостей {news_id: category}
news_category_dict = news_df.set_index('news_id')['category'].to_dict()

# 2. Глобальная популярность и CTR (уже есть из train, зафиксируем еще раз)
train_clicks, train_impressions = extract_item_stats(train_df)

# --- ОБНОВЛЕННАЯ ФУНКЦИЯ СБОРКИ ДАТАСЕТА С НОВЫМИ ФИЧАМИ ---
def prepare_advanced_dataset(df, is_train=True):
    rows = []

    # Для прогресс-бара
    df_iter = tqdm.tqdm(df.iterrows(), total=len(df), desc="Processing sessions")

    for idx, row in df_iter:
        imp_str = row['impressions']
        if pd.isna(imp_str): continue

        session_time = pd.to_datetime(row['time'], format='%m/%d/%Y %I:%M:%S %p')

        # --- ЮЗЕР ФИЧИ (ПЕРСОНАЛИЗАЦИЯ) ---
        history = str(row['history']).split() if pd.notna(row['history']) else []
        history_len = len(history)

        # Получаем категории из истории
        hist_categories = [news_category_dict.get(nid) for nid in history if news_category_dict.get(nid)]
        hist_cat_counts = Counter(hist_categories)
        total_hist_cats = len(hist_categories)

        # Топ-1 категория пользователя
        fav_cat = hist_cat_counts.most_common(1)[0][0] if hist_cat_counts else None

        items = imp_str.split()
        for pos, item in enumerate(items):
            parts = item.split('-')
            news_id = parts[0]
            # В train используем метки, в test - только кандидатов (pointwise approach)
            label = int(parts[1]) if is_train or '-' in item else None

            # --- ИТЕМ ФИЧИ ---
            item_cat = news_category_dict.get(news_id)

            clicks = train_clicks.get(news_id, 0)
            imps = train_impressions.get(news_id, 0)
            ctr = clicks / imps if imps > 0 else 0.0

            # --- ВЗАИМОДЕЙСТВИЕ (ADVANCED) ---
            # 1. Категориальное совпадение
            is_fav_cat = (item_cat == fav_cat) if fav_cat and item_cat else False

            # 2. Доля этой категории в истории
            cat_pref_score = (hist_cat_counts.get(item_cat, 0) / total_hist_cats) if total_hist_cats > 0 and item_cat else 0.0

            rows.append({
                'session_id': idx,
                'news_id': news_id,
                # Старые фичи
                'position_in_impression': pos,
                'history_len': history_len,
                'train_clicks': clicks,
                'train_impressions': imps,
                'train_ctr': ctr,
                # Категориальные (нужно указать CatBoost, что это категории)
                'item_category': item_cat if item_cat else "UNKNOWN",
                'user_fav_category': fav_cat if fav_cat else "UNKNOWN",
                # Новые advanced фичи
                'is_fav_category_match': int(is_fav_cat),
                'category_preference_score': cat_pref_score,
                # Таргет
                'target': label
            })

    return pd.DataFrame(rows)


# Очищаем датафреймы от случайно попавших заголовков
train_df = train_df[train_df['time'] != 'time'].copy()
test_df = test_df[test_df['time'] != 'time'].copy()


# Сборка датасетов (может занять время, т.к. логика усложнилась)
print("\nСборка ADVANCED обучающего датасета (Train)...")
X_train_adv_df = prepare_advanced_dataset(train_df, is_train=True)
# Точечно удаляем NaN в таргете, как в прошлый раз
X_train_adv_df.dropna(subset=['target'], inplace=True)
X_train_adv_df['target'] = X_train_adv_df['target'].astype(int)

print(f"\nAdvanced Обучающих строк: {len(X_train_adv_df)}")

print("\nСборка ADVANCED тестового датасета (Test)...")
# На тесте метки не используем для pointwise inference
X_test_adv_df = prepare_advanced_dataset(test_df, is_train=False)
# Для расчета метрик нам понадобятся метки, извлечем их отдельно
# (в нашей функции prepare_advanced_dataset мы заложили логику is_train)
# но для pointwise inference на тесте нам таргет в датасете не нужен.
# А для расчета NDCG нам нужны реальные клики. Восстановим target в X_test_adv_df из исходного test_df 'impressions'
# так как is_train=False занулил label в rows.append

# Восстановление таргета для теста (Pointwise inference approach):
print("Восстановление таргета для тестового датасета...")
# Создаем маппинг { (session_id, news_id) : label } из сырого test_df
test_target_map = {}
for idx, row in tqdm.tqdm(test_df.iterrows(), total=len(test_df), desc="Mapping test labels"):
    imp_str = row['impressions']
    if pd.isna(imp_str): continue
    for item in imp_str.split():
        parts = item.split('-')
        test_target_map[(idx, parts[0])] = int(parts[1])

# Применяем маппинг к advanced тестовому датасету
X_test_adv_df['target'] = X_test_adv_df.apply(lambda row: test_target_map.get((row['session_id'], row['news_id']), 0), axis=1)

print(f"Advanced Тестовых строк: {len(X_test_adv_df)}")

Подготовка вспомогательных словарей...

Сборка ADVANCED обучающего датасета (Train)...


Processing sessions: 100%|██████████| 126695/126695 [00:38<00:00, 3283.78it/s]



Advanced Обучающих строк: 4621015

Сборка ADVANCED тестового датасета (Test)...


Processing sessions: 100%|██████████| 30270/30270 [00:08<00:00, 3507.15it/s]


Восстановление таргета для тестового датасета...


Mapping test labels: 100%|██████████| 30270/30270 [00:03<00:00, 10065.80it/s]


Advanced Тестовых строк: 1222429


In [13]:
X_train_adv_df.to_csv('../data/X_train_adv_df.tsv', sep='\t')
X_test_adv_df.to_csv('../data/X_test_adv_df.tsv', sep='\t')

In [15]:
from catboost import CatBoostClassifier, Pool


# --- НАЧАЛО БЛОКА ОБУЧЕНИЯ ADVANCED МОДЕЛИ ---

# 1. Выбор фичей для Advanced модели (строго по списку)
feature_cols_adv = [
    # Старые числовые фичи
    'position_in_impression', 'history_len', 'train_clicks', 'train_impressions', 'train_ctr',
    # Новые advanced фичи (персонализация и совпадения)
    'is_fav_category_match', 'category_preference_score',
    # Новые КАТЕГОРИАЛЬНЫЕ фичи (их нужно указать CatBoost)
    'item_category', 'user_fav_category'
]

X_train_adv = X_train_adv_df[feature_cols_adv]
y_train_adv = X_train_adv_df['target'] # Таргет гарантированно без NaN и int

X_test_adv = X_test_adv_df[feature_cols_adv]
# Тагет на тесте восстановлен в X_test_adv_df['target']

# Указываем, какие колонки являются категориальными
cat_features_indices = [7, 8] # Индексы 'item_category' и 'user_fav_category' в feature_cols_adv

# Рассчитываем веса для Logloss (дисбаланс ~4%)
pos_count = y_train_adv.sum()
neg_count = len(y_train_adv) - pos_count
scale_weight_adv = neg_count / pos_count
print(f" scale_pos_weight_adv = {scale_weight_adv:.2f}")

# 2. Подготовка Pool данных (ускоряет обучение с категориями)
train_pool_adv = Pool(X_train_adv, y_train_adv, cat_features=cat_features_indices)

# 3. Обучение Advanced модели (те же параметры для честности)
print("Начинаем обучение Advanced CatBoostClassifier (с персонализацией)...")
model_adv = CatBoostClassifier(
    iterations=500,
    depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_weight_adv,
    loss_function='Logloss',
    verbose=100,
    random_seed=42
)

# Обучаем модель, передавая Pool
model_adv.fit(train_pool_adv)
print("Advanced Модель обучена!")

# 4. Инференс на advanced тесте (pointwise CTR prediction)
print("Прогнозирование на advanced тестовых данных...")
# При инференсе на Pool переводить не обязательно, CatBoost сам поймет категории,
# если датасет и feature_cols совпадают с обучением.
preds_proba_adv = model_adv.predict_proba(X_test_adv)[:, 1]
# Сохраняем новые скоры
X_test_adv_df['score_adv'] = preds_proba_adv

# 5. Расчет метрик ранжирования (по сессиям)
# Функции NDCG и Recall уже должны быть определены, но зафиксируем их еще раз:

def recall_at_k(actual, predicted, k=5):
    predicted_at_k = predicted[:k]
    hits = len(set(actual) & set(predicted_at_k))
    return hits / len(actual) if len(actual) > 0 else 0.0

def ndcg_at_k(actual, predicted, k=5):
    predicted_at_k = predicted[:k]
    dcg = 0.0
    for i, p in enumerate(predicted_at_k):
        if p in actual:
            dcg += 1.0 / np.log2(i + 2)
    idcg = sum([1.0 / np.log2(i + 2) for i in range(min(len(actual), k))])
    return dcg / idcg if idcg > 0 else 0.0

model_ndcg_adv, model_recall_adv = [], []

test_sessions_adv = X_test_adv_df.groupby('session_id')

print("Расчет метрик ранжирования для Advanced модели...")
for _, session_data in tqdm.tqdm(test_sessions_adv, desc="Calculating NDCG/Recall"):
    # Таргет восстановлен, берем клики
    actual_clicks = session_data[session_data['target'] == 1]['news_id'].tolist()
    if not actual_clicks: continue

    # Сортируем по НОВОМУ скору ('score_adv')
    ranked_recommendations = session_data.sort_values(by='score_adv', ascending=False)['news_id'].tolist()

    # Считаем метрики для топ-5
    model_ndcg_adv.append(ndcg_at_k(actual_clicks, ranked_recommendations, k=5))
    model_recall_adv.append(recall_at_k(actual_clicks, ranked_recommendations, k=5))

# --- КОНЕЦ БЛОКА ОБУЧЕНИЯ ---

# Выводим сравнение результатов:
print("\n=== СРАВНЕНИЕ РЕЗУЛЬТАТОВ (на Test set) ===")
# Старые результаты (зафиксированы ранее)
print(f"1. Random Baseline:    NDCG@5 = 0.2028 | Recall@5 = 0.3078")
print(f"2. Popularity Baseline: NDCG@5 = 0.2168 | Recall@5 = 0.3253")
print(f"3. CatBoost (base features): NDCG@5 = 0.2319 | Recall@5 = 0.3408")
# Новые результаты (рассчитанные только что)
print(f"4. CatBoost (ADVANCED features): NDCG@5 = {np.mean(model_ndcg_adv):.4f} | Recall@5 = {np.mean(model_recall_adv):.4f}")

 scale_pos_weight_adv = 23.38
Начинаем обучение Advanced CatBoostClassifier (с персонализацией)...
0:	learn: 0.6714938	total: 1.76s	remaining: 14m 40s
100:	learn: 0.5628707	total: 2m 9s	remaining: 8m 30s
200:	learn: 0.5604357	total: 4m 4s	remaining: 6m 3s
300:	learn: 0.5587393	total: 6m 1s	remaining: 3m 59s
400:	learn: 0.5574050	total: 8m 4s	remaining: 1m 59s
499:	learn: 0.5562913	total: 10m 10s	remaining: 0us
Advanced Модель обучена!
Прогнозирование на advanced тестовых данных...
Расчет метрик ранжирования для Advanced модели...


Calculating NDCG/Recall: 100%|██████████| 30270/30270 [00:32<00:00, 931.12it/s] 


=== СРАВНЕНИЕ РЕЗУЛЬТАТОВ (на Test set) ===
1. Random Baseline:    NDCG@5 = 0.2028 | Recall@5 = 0.3078
2. Popularity Baseline: NDCG@5 = 0.2168 | Recall@5 = 0.3253
3. CatBoost (base features): NDCG@5 = 0.2319 | Recall@5 = 0.3408
4. CatBoost (ADVANCED features): NDCG@5 = 0.2541 | Recall@5 = 0.3721
